In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv("/home/admin1/Downloads/train.csv")

In [5]:
df

,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."
...,...,...,...
119995,1,Pakistan's Musharraf Says Won't Quit as Army C...,KARACHI (Reuters) - Pakistani President Perve...
119996,2,Renteria signing a top-shelf deal,Red Sox general manager Theo Epstein acknowled...
119997,2,Saban not going to Dolphins yet,The Miami Dolphins will put their courtship of...
119998,2,Today's NFL games,PITTSBURGH at NY GIANTS Time: 1:30 p.m. Line: ...


In [7]:
df.isnull().sum()

Class Index    0
Title          0
Description    0
dtype: int64

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   Class Index  120000 non-null  int64 
 1   Title        120000 non-null  object
 2   Description  120000 non-null  object
dtypes: int64(1), object(2)
memory usage: 2.7+ MB


In [17]:
import nltk
import re

In [19]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [21]:
lemma = WordNetLemmatizer()

In [29]:
corpus_title = []
corpus_desc = []
for i in range(len(df)):
    for col,corpus in [("Title",corpus_title) , ("Description",corpus_desc)]:
        review = re.sub('[^a-zA-Z]',' ',df[col][i])
        review = review.lower().split()
        review = [lemma.lemmatize(word) for word in review if word.lower() not in stopwords.words('english')]
        corpus.append(" ".join(review))

In [31]:
corpus_title

['wall st bear claw back black reuters',
 'carlyle look toward commercial aerospace reuters',
 'oil economy cloud stock outlook reuters',
 'iraq halt oil export main southern pipeline reuters',
 'oil price soar time record posing new menace u economy afp',
 'stock end near year low reuters',
 'money fund fell latest week ap',
 'fed minute show dissent inflation usatoday com',
 'safety net forbes com',
 'wall st bear claw back black',
 'oil economy cloud stock outlook',
 'need opec pump iran gov',
 'non opec nation output purnomo',
 'google ipo auction rocky start',
 'dollar fall broadly record trade gap',
 'rescuing old saver',
 'kid rule back school',
 'market head toward value fund',
 'u trade deficit swell june',
 'shell could target total',
 'google ipo face playboy slip',
 'eurozone economy keep growing',
 'expansion slows japan',
 'rand fall shock sa rate cut',
 'car price across board',
 'south korea lower interest rate',
 'google auction begin friday',
 'hp share tumble profit 

In [33]:
corpus_desc

['reuters short seller wall street dwindling band ultra cynic seeing green',
 'reuters private investment firm carlyle group reputation making well timed occasionally controversial play defense industry quietly placed bet another part market',
 'reuters soaring crude price plus worry economy outlook earnings expected hang stock market next week depth summer doldrums',
 'reuters authority halted oil export flow main pipeline southern iraq intelligence showed rebel militia could strike infrastructure oil official said saturday',
 'afp tearaway world oil price toppling record straining wallet present new economic menace barely three month u presidential election',
 'reuters stock ended slightly higher friday stayed near low year oil price surged past barrel offsetting positive outlook computer maker dell inc dell',
 'ap asset nation retail money market mutual fund fell billion latest week trillion investment company institute said thursday',
 'usatoday com retail sale bounced back bit jul

In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_title = TfidfVectorizer(max_features=5000)
tfidf_desc = TfidfVectorizer(max_features=5000)

x_title = tfidf_title.fit_transform(corpus_title)
x_desc  = tfidf_desc.fit_transform(corpus_desc)

In [53]:
tfidf_title.vocabulary_

{'wall': 4815,
 'st': 4187,
 'bear': 380,
 'back': 311,
 'black': 451,
 'reuters': 3680,
 'look': 2544,
 'toward': 4554,
 'commercial': 892,
 'aerospace': 60,
 'oil': 3026,
 'economy': 1362,
 'cloud': 844,
 'stock': 4233,
 'outlook': 3086,
 'iraq': 2244,
 'halt': 1928,
 'export': 1510,
 'main': 2594,
 'southern': 4123,
 'pipeline': 3241,
 'price': 3369,
 'soar': 4089,
 'time': 4512,
 'record': 3558,
 'new': 2931,
 'afp': 66,
 'end': 1412,
 'near': 2909,
 'year': 4976,
 'low': 2562,
 'money': 2804,
 'fund': 1749,
 'fell': 1590,
 'latest': 2438,
 'week': 4858,
 'ap': 174,
 'fed': 1581,
 'minute': 2768,
 'show': 3989,
 'inflation': 2171,
 'usatoday': 4717,
 'com': 880,
 'safety': 3794,
 'net': 2923,
 'forbes': 1691,
 'need': 2914,
 'opec': 3042,
 'pump': 3443,
 'iran': 2242,
 'gov': 1849,
 'non': 2968,
 'nation': 2897,
 'output': 3087,
 'google': 1844,
 'ipo': 2239,
 'auction': 277,
 'rocky': 3732,
 'start': 4205,
 'dollar': 1266,
 'fall': 1543,
 'trade': 4564,
 'gap': 1770,
 'old': 3030,

In [47]:
from scipy.sparse import hstack           ### Sparse matrix conversion to one matrix
             
x = hstack([x_title, x_desc])
y = df['Class Index']

In [55]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,train_size=0.8)

### Multinomial Naive Bayes

In [57]:
from sklearn.naive_bayes import MultinomialNB
nb= MultinomialNB()

In [59]:
nb.fit(x_train,y_train)

MultinomialNB()

In [61]:
y_pred = nb.predict(x_test)

In [68]:
from sklearn.metrics import accuracy_score,classification_report
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.8979583333333333


In [70]:
print("Classification Report:", classification_report(y_test, y_pred))

Classification Report:               precision    recall  f1-score   support

           1       0.91      0.89      0.90      5986
           2       0.94      0.97      0.96      5995
           3       0.87      0.86      0.86      5985
           4       0.87      0.87      0.87      6034

    accuracy                           0.90     24000
   macro avg       0.90      0.90      0.90     24000
weighted avg       0.90      0.90      0.90     24000



### Random Forest Classification

In [75]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()

In [77]:
rf.fit(x_train,y_train)

RandomForestClassifier()

In [79]:
y_pred1 = rf.predict(x_test)

In [81]:
from sklearn.metrics import accuracy_score,classification_report
print("Accuracy:", accuracy_score(y_test, y_pred1))

Accuracy: 0.8873333333333333


In [83]:
print("Classification Report:", classification_report(y_test, y_pred1))

Classification Report:               precision    recall  f1-score   support

           1       0.90      0.89      0.89      5986
           2       0.92      0.96      0.94      5995
           3       0.86      0.85      0.86      5985
           4       0.87      0.85      0.86      6034

    accuracy                           0.89     24000
   macro avg       0.89      0.89      0.89     24000
weighted avg       0.89      0.89      0.89     24000



### Logistic Regression 

In [86]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()

In [88]:
lr.fit(x_train,y_train)

/home/admin1/anaconda3/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [90]:
y_pred2 = lr.predict(x_test)

In [92]:
from sklearn.metrics import accuracy_score,classification_report
print("Accuracy:", accuracy_score(y_test, y_pred2))

Accuracy: 0.910375


In [94]:
print("Classification Report:", classification_report(y_test, y_pred2))

Classification Report:               precision    recall  f1-score   support

           1       0.91      0.90      0.91      5986
           2       0.96      0.97      0.96      5995
           3       0.88      0.88      0.88      5985
           4       0.89      0.89      0.89      6034

    accuracy                           0.91     24000
   macro avg       0.91      0.91      0.91     24000
weighted avg       0.91      0.91      0.91     24000

